# CyberGuard AI - Email & SMS NLP Training
This completely self-contained notebook downloads 5,500+ real-world Spam and Safe text messages, processes them, and trains a DistilBERT NLP model on your Colab GPU.
**Just click Run All!**

In [ ]:
!pip install transformers torch scikit-learn pandas tqdm

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- 1. Fetch NLP Data ---
print("Downloading UCI SMS & Email Spam Dataset...")
try:
    # Public TSV containing 5,500 real-world messages
    url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
    df = pd.read_table(url, header=None, names=['label', 'message'])
    
    # Map 'ham' (safe) to 0 and 'spam' (phishing/malicious) to 1
    df['target'] = df.label.map({'ham': 0, 'spam': 1})
except Exception as e:
    print("Download failed, creating fallback data...", e)
    df = pd.DataFrame({
        'message': ["URGENT: Your account is locked! Click here.", "Hey man, want to grab lunch tomorrow?", "Claim your $500 Amazon gift card now!", "Did you finish the quarterly report?"],
        'target': [1, 0, 1, 0]
    })

print(f"\nData loaded: {len(df)} Text Messages ({(df['target']==1).sum()} Phishing/Spam, {(df['target']==0).sum()} Safe)")

# --- 2. NLP Transformer Training ---
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return { "input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(self.labels[idx], dtype=torch.long) }

print("\nLoading NLP Engine (DistilBERT)...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to("cuda")

train_ds = TextDataset(df["message"].values, df["target"].values, tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
model.train()

print("\nStarting NLP Training on NVIDIA GPU! This will take ~2 minutes...")
EPOCHS = 3 # We do 3 epochs because the dataset is smaller (5,500 samples)
for epoch in range(EPOCHS):
    total_loss = 0
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"].to("cuda"), 
            attention_mask=batch["attention_mask"].to("cuda"), 
            labels=batch["labels"].to("cuda")
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 100 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")
    print(f"-> Epoch {epoch+1} Average Loss: {total_loss/len(train_loader):.4f}\n")

# --- 3. Save Model ---
export_dir = "finetuned_email_transformer"
os.makedirs(export_dir, exist_ok=True)
model.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)
print(f"\n[Success] Training complete! Model saved to the '{export_dir}' folder.")
print(f"Right-click the '{export_dir}' folder in the Colab sidebar and click Download!")
